# Introduction to Autograd
Use PyTorch automatic differentiation to build a computation graph, calculate gradients, inspect accumulation, and clear gradients safely.

In [ ]:
import torch

torch.manual_seed(42)

## Track Operations with `requires_grad`
A tensor records operations for automatic differentiation when `requires_grad=True`.

In [ ]:
x = torch.tensor([2.0, 3.0], requires_grad=True)
squared = x.pow(2)
loss = squared.sum()

print(f"x.is_leaf: {x.is_leaf}")
print(f"squared.grad_fn: {squared.grad_fn}")
print(f"loss.grad_fn: {loss.grad_fn}")

## Calculate Gradients with `backward()`
For `loss = x₁² + x₂²`, the derivative with respect to each value is `2x`.

In [ ]:
loss.backward()
print(f"x: {x}")
print(f"x.grad: {x.grad}")

## Gradients Accumulate
PyTorch adds new gradients to the existing `.grad` value. This is useful for intentional gradient accumulation, but ordinary training steps clear gradients first.

In [ ]:
weight = torch.tensor(2.0, requires_grad=True)

first_loss = (weight * 3).pow(2)
first_loss.backward()
print(f"After first backward: {weight.grad}")

second_loss = (weight * 3).pow(2)
second_loss.backward()
print(f"After second backward: {weight.grad}")

## Clear Gradients
For an individual tensor, clear its gradient in place. Optimizers normally use `optimizer.zero_grad()` for model parameters.

In [ ]:
weight.grad.zero_()
print(f"Cleared gradient: {weight.grad}")

## Use Autograd in an Optimization Step
The standard order is: clear gradients, run the forward pass, calculate loss, call `backward()`, and update the parameters.

In [ ]:
model = torch.nn.Linear(2, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
features = torch.tensor([[1.0, 2.0], [2.0, 1.0]])
targets = torch.tensor([[1.0], [1.0]])

optimizer.zero_grad()
predictions = model(features)
training_loss = torch.nn.functional.mse_loss(predictions, targets)
training_loss.backward()

print(f"Loss: {training_loss.item():.4f}")
print(f"Weight gradient: {model.weight.grad}")
optimizer.step()

## Disable Gradient Tracking When It Is Not Needed
Evaluation and inference do not update parameters. Use a no-gradient context to avoid recording an unnecessary graph.

In [ ]:
with torch.no_grad():
    evaluation_predictions = model(features)

print(f"Tracks gradients: {evaluation_predictions.requires_grad}")

## Quick Review
- `requires_grad=True` enables operation tracking.
- `backward()` applies the chain rule and stores gradients on leaf tensors.
- Gradients accumulate until they are cleared.
- A normal training step clears gradients before the next backward pass.
- Disable gradient tracking for evaluation and inference.